# Entity Sentiment Analysis

This notebook explains how the sentiment analyzer works in the wigglnet data engine.

## Overview

The sentiment analyzer identifies **named entities** (organizations, people, products, locations) in article text and determines the **sentiment** expressed toward each entity.

### Key Features
- Combines **spaCy NER** (Named Entity Recognition) with **RoBERTa** sentiment classification
- Analyzes sentiment at the entity level, not just document level
- Returns confidence scores and mention counts
- Processes articles in the same format as the keyword extractor

### Use Cases
- Track brand sentiment in news articles
- Monitor public perception of companies, products, or people
- Analyze competitor sentiment
- Identify trends in entity coverage (positive vs negative)

## Architecture

The sentiment analyzer uses a two-stage pipeline:

```
Article Text
    ↓
┌─────────────────────────────────────┐
│  Stage 1: Named Entity Recognition  │
│  (spaCy en_core_web_md)             │
└─────────────────────────────────────┘
    ↓
Entities: ["Apple", "Microsoft", "Tim Cook", ...]
    ↓
┌─────────────────────────────────────┐
│  Stage 2: Sentiment Classification  │
│  (RoBERTa cardiffnlp/twitter-roberta)│
└─────────────────────────────────────┘
    ↓
Results: [{entity: "Apple", sentiment: "positive", confidence: 0.89, ...}]
```

### Why Two Models?

1. **spaCy** - Fast, rule-based + statistical NER
   - Identifies entities (WHO/WHAT the text is about)
   - Categorizes entity types (ORG, PERSON, PRODUCT, etc.)
   - Processes entire documents in milliseconds

2. **RoBERTa** - Deep learning sentiment classifier
   - Understands contextual sentiment (HOW entities are discussed)
   - Pre-trained on Twitter data for robust social sentiment
   - Returns confidence scores for positive/neutral/negative

## Setup and Imports

First, let's import the sentiment analyzer and ensure models are downloaded.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from app.keywords.tools.sentiment_analyzer import (
    ensure_models,
    analyze_entity_sentiment,
    add_sentiment_to_articles
)

# Download models if needed (only runs once)
print("Ensuring models are downloaded...")
ensure_models()
print("✓ Models ready!")

## Stage 1: Named Entity Recognition (spaCy)

spaCy identifies and categorizes entities in text.

### Entity Types Tracked

| Type | Description | Example |
|------|-------------|----------|
| ORG | Organizations | Apple, Microsoft, UN |
| PERSON | People | Tim Cook, Elon Musk |
| PRODUCT | Products/Services | iPhone, ChatGPT |
| GPE | Geopolitical entities | USA, California, Paris |
| NORP | Nationalities/groups | American, Republican |

Let's see spaCy NER in action:

In [ ]:
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_md")

sample_text = """
Apple reported record earnings this quarter, exceeding analyst expectations.
Tim Cook credited strong iPhone sales and the growing services division.
Meanwhile, Microsoft announced sweeping layoffs affecting thousands of employees.
"""

doc = nlp(sample_text)

print("Entities found by spaCy:\n")
for ent in doc.ents:
    print(f"  {ent.text:<20} → {ent.label_:<10} ({spacy.explain(ent.label_)})")

## Stage 2: Sentiment Classification (RoBERTa)

RoBERTa analyzes the emotional tone of sentences containing entities.

### How RoBERTa Works

1. **Tokenization** - Breaks text into subword tokens
2. **Contextual Embeddings** - Converts tokens to vectors that capture meaning
3. **Classification** - Predicts sentiment label with confidence score

### Model Details
- **Model**: `cardiffnlp/twitter-roberta-base-sentiment-latest`
- **Training**: Fine-tuned on Twitter data (~124M tweets)
- **Output**: 3 classes (positive, neutral, negative) + confidence (0-1)

Let's test RoBERTa on individual sentences:

In [ ]:
from transformers import pipeline

# Load RoBERTa sentiment classifier
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True
)

sentences = [
    "Apple reported record earnings this quarter.",
    "Microsoft announced sweeping layoffs.",
    "The company continues to operate normally."
]

print("RoBERTa Sentiment Analysis:\n")
for sentence in sentences:
    result = classifier(sentence)[0]
    label = result['label']
    score = result['score']
    
    # Map LABEL_0/1/2 to readable names
    sentiment_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
    sentiment = sentiment_map.get(label, label)
    
    print(f"  Sentence: {sentence}")
    print(f"  → {sentiment.upper()} (confidence: {score:.3f})\n")

## The Complete Pipeline

Now let's see how the two stages work together to analyze entity-level sentiment.

### Algorithm Steps

1. **Extract entities** - Use spaCy to find all named entities
2. **Tokenize sentences** - Split text into sentences
3. **Match entities to sentences** - Find which sentences mention each entity
4. **Classify sentiment** - Run RoBERTa on sentences containing the entity
5. **Aggregate scores** - Average confidence scores per entity
6. **Rank by mentions** - Sort entities by how often they appear

### Visual Example

```
Text: "Apple reported record earnings. Tim Cook praised the team."

Step 1 (spaCy NER):
  Entities: ["Apple", "Tim Cook"]

Step 2 (Tokenize):
  Sentences: [
    "Apple reported record earnings.",
    "Tim Cook praised the team."
  ]

Step 3 (Match):
  "Apple" → ["Apple reported record earnings."]
  "Tim Cook" → ["Tim Cook praised the team."]

Step 4 (RoBERTa):
  "Apple reported record earnings." → positive (0.89)
  "Tim Cook praised the team." → positive (0.92)

Step 5 (Results):
  Apple: positive, confidence=0.89, mentions=1
  Tim Cook: positive, confidence=0.92, mentions=1
```

## Full Example

Let's analyze a realistic article with multiple entities and mixed sentiment.

In [ ]:
article_text = """
Apple reported record earnings this quarter, exceeding analyst expectations.
Tim Cook credited strong iPhone sales and the growing services division.
The company's innovation in AI has impressed industry watchers.

Meanwhile, Microsoft announced sweeping layoffs affecting thousands of employees.
Satya Nadella defended the decision, citing a need to refocus on AI investment.
Despite the cuts, Microsoft stock rose on optimism about future growth.

The European Union has raised concerns about both companies' data practices,
with regulators signaling potential fines under the Digital Markets Act.
Privacy advocates have praised the EU's strong stance on corporate accountability.
"""

# Run the complete analysis
results = analyze_entity_sentiment(article_text, top_n=10)

print(f"Found {len(results)} entities with sentiment:\n")
print(f"{'Entity':<20} {'Type':<10} {'Sentiment':<12} {'Confidence':<12} {'Mentions'}")
print("="*70)

for r in results:
    print(
        f"{r['entity']:<20} {r['entity_type']:<10} {r['sentiment']:<12} "
        f"{r['confidence']:<12.3f} {r['mentions']}"
    )

## Understanding the Results

Let's break down what each field means:

### Result Fields

| Field | Type | Description |
|-------|------|-------------|
| `entity` | str | The entity name as it appears in text |
| `entity_type` | str | spaCy category (ORG, PERSON, etc.) |
| `sentiment` | str | Dominant sentiment (positive/neutral/negative) |
| `confidence` | float | RoBERTa confidence score (0-1) |
| `mentions` | int | Number of sentences mentioning this entity |

### Interpreting Confidence Scores

- **0.90 - 1.00**: Very confident - clear sentiment
- **0.70 - 0.89**: Confident - reliable sentiment
- **0.50 - 0.69**: Moderate - some uncertainty
- **< 0.50**: Low confidence - mixed or unclear sentiment

### Sentiment Aggregation Logic

When an entity appears in multiple sentences with different sentiments:

1. RoBERTa scores each sentence separately
2. Scores are grouped by sentiment class (pos/neu/neg)
3. The class with the highest total score wins
4. Confidence is the average of winning class scores

Example:

In [ ]:
# Example: Entity with mixed sentiment
mixed_text = """
Tesla unveiled its impressive new cybertruck design today.
However, Tesla faced criticism over production delays.
Despite challenges, Tesla stock remained strong.
"""

result = analyze_entity_sentiment(mixed_text, top_n=1)

print("Entity with mixed sentiment:\n")
print(f"Entity: {result[0]['entity']}")
print(f"Sentiment: {result[0]['sentiment']}")
print(f"Confidence: {result[0]['confidence']:.3f}")
print(f"Mentions: {result[0]['mentions']}")
print("\nNote: Sentiment is determined by the dominant trend across all mentions.")

## Integration with Article Pipeline

The sentiment analyzer mirrors the interface of the keyword extractor, making it easy to integrate into the scraping pipeline.

In [ ]:
# Example article dict (same format as web scraper produces)
articles = [
    {
        'title': 'Tech Giants Report Q4 Earnings',
        'link': 'https://example.com/article1',
        'paragraphs': [
            'Apple reported record earnings this quarter.',
            'Tim Cook praised the engineering team.',
            'Microsoft also exceeded expectations.'
        ]
    },
    {
        'title': 'EU Regulatory Action',
        'link': 'https://example.com/article2',
        'paragraphs': [
            'The European Union imposed fines on tech companies.',
            'Regulators cited privacy violations.',
        ]
    }
]

# Add sentiment analysis to all articles
articles_with_sentiment = add_sentiment_to_articles(articles, top_n=5, verbose=True)

# View results
print("\n" + "="*70)
for article in articles_with_sentiment:
    print(f"\nArticle: {article['title']}")
    if article.get('entity_sentiment'):
        for entity in article['entity_sentiment']:
            print(f"  - {entity['entity']}: {entity['sentiment']} (conf: {entity['confidence']:.2f})")
    else:
        print("  No entities detected")

## Database Storage

Entity sentiment results are stored in the database alongside articles:

### Tables

1. **ENTITIES**
   - `ENTITY_ID` (primary key)
   - `ENTITY_TEXT` (e.g., "Apple")
   - `ENTITY_TYPE` (e.g., "ORG")

2. **ARTICLE_ENTITY_SENTIMENT**
   - `ARTICLE_ID` (foreign key)
   - `ENTITY_ID` (foreign key)
   - `SENTIMENT` (positive/neutral/negative)
   - `CONFIDENCE` (0-1)
   - `MENTIONS` (count)

This allows querying like:
- "Show all articles with negative sentiment about Apple"
- "Track sentiment trend for Microsoft over time"
- "Find most-mentioned entities this month"

## Performance Considerations

### Speed
- spaCy NER: ~100-500 articles/second
- RoBERTa classification: ~10-50 sentences/second (GPU can 10x this)
- Total pipeline: ~5-10 articles/second on CPU

### Memory
- spaCy model: ~100MB RAM
- RoBERTa model: ~500MB RAM
- Total footprint: ~600-800MB

### First Run
Models are downloaded automatically on first use:
- spaCy `en_core_web_md`: ~40MB download
- RoBERTa model: ~500MB download

### Optimization Tips
1. Process articles in batches
2. Use GPU if available (set `device=0` in pipeline)
3. Filter entities by type before sentiment analysis
4. Cache results in database to avoid reprocessing

## Comparison: Document-level vs Entity-level Sentiment

Traditional sentiment analysis scores entire documents. Our approach is more granular.

### Example Article

"Apple reported strong earnings while Microsoft announced layoffs."

**Document-level**: Mixed/Neutral (one positive + one negative = neutral)

**Entity-level** (our approach):
- Apple: Positive
- Microsoft: Negative

This granularity is crucial for:
- Competitive analysis
- Brand monitoring
- Understanding nuanced coverage

## Limitations & Future Improvements

### Current Limitations

1. **Sarcasm/Irony**: RoBERTa can misinterpret sarcastic statements
2. **Entity Disambiguation**: "Apple" could mean company or fruit
3. **Complex Sentiments**: Nuanced opinions may be oversimplified
4. **English Only**: Current models only support English text

### Potential Improvements

1. **Aspect-based Sentiment**: Analyze specific aspects (price, quality, service)
2. **Temporal Tracking**: Monitor sentiment changes over time
3. **Emotion Detection**: Beyond pos/neg - detect joy, anger, fear, etc.
4. **Cross-lingual**: Add support for Spanish, French, etc.
5. **Custom Training**: Fine-tune on domain-specific data (e.g., home design articles)

## Summary

The sentiment analyzer combines two powerful NLP models:

1. **spaCy** identifies WHO/WHAT (entities)
2. **RoBERTa** determines HOW they're discussed (sentiment)

This entity-level approach provides granular insights that document-level sentiment misses.

### Key Takeaways
- ✅ Fast and accurate entity detection
- ✅ State-of-the-art sentiment classification
- ✅ Confidence scores for quality assessment
- ✅ Easy integration with article pipeline
- ✅ Database-ready output format

### Next Steps
- See `web_scraper.py` for integration example
- Check `db_uploader.py` for database storage
- Explore SQL queries for sentiment analysis over time